## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Before we start - some setup:


Please visit Sendgrid at: https://sendgrid.com/

(Sendgrid is a Twilio company for sending emails.)

If SendGrid gives you problems, see the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

Please set up an account - it's free! (at least, for me, right now).

Once you've created an account, click on:

Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:

`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:

Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.


In [12]:
from dotenv import load_dotenv
from agents import Runner, Agent, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [13]:
load_dotenv(override=True)

True

In [5]:

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("cejhei@gmail.com")  # Change to your verified sender
    to_email = To("heginajason@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


In [6]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [14]:
sales_agent1 =  Agent(
    name="Professional Sale Agent",
    model="gpt-4o-mini",
    instructions=instructions1
)

sales_agent2 =  Agent(
    name="Engaging Sale Agent",
    model="gpt-4o-mini",
    instructions=instructions2
)

sales_agent3 =  Agent(
    name="Busy Sale Agent",
    model="gpt-4o-mini",
    instructions=instructions3
)



In [15]:
# stream results

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

Navigating the complexities of SOC 2 compliance can be a daunting task, especially when preparing for audits. At ComplAI, we specialize in simplifying this process for businesses like yours through our AI-driven SaaS tool.

Our platform not only streamlines compliance management but also helps you stay aligned with industry standards, mitigating risks and enhancing your organization’s credibility. With features designed to save time and reduce headaches, we empower teams to focus on what matters most—growing your business.

I would love the opportunity to discuss how ComplAI can specifically assist [Recipient's Company] in achieving and maintaining compliance seamlessly. Are you available for a brief call next week?

Thank you for considering this opportunity. I look forward to your response.

Best regards,

[Your Name]  
[Your Position]  
ComplAI  
[Your Phone Number]  

In [17]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Simplify Your SOC2 Compliance Journey with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m with ComplAI, where we specialize in streamlining the SOC2 compliance process for organizations like yours.

Navigating SOC2 compliance can be a daunting task, often filled with convoluted checklists and tight deadlines. ComplAI’s AI-driven SaaS tool simplifies this process, providing you with seamless preparation for audits while ensuring you meet all necessary compliance requirements efficiently.

Here are a few ways ComplAI can benefit your organization:

- **Automated Documentation:** Reduce manual effort with our intelligent documentation system that aligns with SOC2 mandates.
- **Real-Time Compliance Tracking:** Get continuous oversight of your compliance status, helping you stay audit-ready at all times.
- **Expert-Driven Insights:** Leverage AI to receive personalized recommendations based on your organization’s unique needs.



In [18]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [19]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Is Your SOC2 Compliance Process as Smooth as a Cat on a Roomba? 🐱🤖

Hey [Recipient's Name],

I hope this email lands in your inbox as smoothly as a cat perched on a Roomba! 🐾

Let’s talk about SOC2 compliance. If your current process feels like trying to untangle a set of Christmas lights — frustrating and hair-pulling (and maybe a little colorful) — we need to chat!

At ComplAI, we’ve got a SaaS tool that uses AI to make SOC2 compliance and audits as breezy as a Sunday morning. Think of us as your compliance sidekick, minus the cape but with all the superpowers you need!

Here are a couple of things our clients are raving about:
- Automated documentation? Check! No more late-night memos written in caffeinated desperation.
- Streamlined audit prep! Say goodbye to the confusion that makes IKEA instructions look like child’s play.

Curious to see how we can help you transform compliance from a headache into a well-oiled machine? Let’s schedule a quick call — I 

In [ ]:
#simplifying the boiler plate for creating tools
#by addind a decorator @function_tool from openai's agent package

@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """ #will turn into description. This is a doc string
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("cejhei@gmail.com")  # Change to your verified sender
    to_email = To("heginajason@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [22]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453BABE160>, strict_json_schema=True, is_enabled=True)

In [24]:
# you can also turn an agent into a tool with agent.as_tool
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453BDB04A0>, strict_json_schema=True, is_enabled=True)

In [26]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agen1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agen2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agen3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agen1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agen1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453BD91620>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agen2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agen2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453BD91EE0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agen3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, '

In [27]:
#Planning agent -> Sales manager

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")
message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    results = await Runner.run(sales_manager, message)



In [36]:
#What are handoffs
#control passes across while tools passes back the control. Passing the entire job to the next agent

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [38]:
@function_tool
def send_html_email(subject:str, html_body: str):
    """ Send out an email with the given body and HTML to all sales prospects """ #will turn into description. This is a doc string
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("cejhei@gmail.com")  # Change to your verified sender
    to_email = To("heginajason@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [39]:
tools = [subject_tool, html_tool, send_html_email]
tools


[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453F6114E0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453F6D7BA0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_html_email', description='Send out an email with the given body and HTML to all sale

In [31]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

email_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and sent it" #tells other agent that what this agent is useful for
)

In [40]:
tools = [tool1, tool2, tool3]
handoffs = [email_agent]

print(tools)
print(handoffs)

[FunctionTool(name='sales_agen1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agen1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453BD91620>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agen2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agen2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001453BD91EE0>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agen3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 're

In [41]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
    name="Sales Manager", 
    instructions=sales_manager_instructions, 
    tools=tools, 
    handoffs=handoffs,
    model="gpt-4o-mini"
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)